# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

In [ ]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [ ]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [ ]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [5]:
# Miejsce na rozwiązanie zadania 1

import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

NUMBER_OF_FACTS = 20


def fetch_cat_fact():
    """Pobiera jeden losowy fakt o kocie z API."""
    try:
        response = requests.get(CAT_API_URL, timeout=10)
        response.raise_for_status()
        return response.json().get("fact")
    except requests.RequestException as e:
        return f"Błąd podczas pobierania: {e}"


def fetch_sequential(n):
    """Pobiera fakty sekwencyjnie."""
    facts = []
    start_time = time.time()

    for _ in range(n):
        facts.append(fetch_cat_fact())

    end_time = time.time()
    total_time = end_time - start_time
    return facts, total_time


def fetch_threaded(n, max_workers=5):
    """Pobiera fakty wielowątkowo przy użyciu ThreadPoolExecutor."""
    start_time = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        facts = list(executor.map(lambda _: fetch_cat_fact(), range(n)))

    end_time = time.time()
    total_time = end_time - start_time
    return facts, total_time


def main():
    print("=== Pobieranie sekwencyjne ===")
    sequential_facts, sequential_time = fetch_sequential(NUMBER_OF_FACTS)

    for i, fact in enumerate(sequential_facts, 1):
        print(f"{i}. {fact}")

    print(f"\nCzas sekwencyjny: {sequential_time:.2f} s\n")

    print("=== Pobieranie wielowątkowe ===")
    threaded_facts, threaded_time = fetch_threaded(NUMBER_OF_FACTS, max_workers=5)

    for i, fact in enumerate(threaded_facts, 1):
        print(f"{i}. {fact}")

    print(f"\nCzas wielowątkowy: {threaded_time:.2f} s\n")

    print("=== Porównanie ===")
    if threaded_time < sequential_time:
        print(f"Wersja wielowątkowa była szybsza o {sequential_time - threaded_time:.2f} s.")
    elif threaded_time > sequential_time:
        print(f"Wersja sekwencyjna była szybsza o {threaded_time - sequential_time:.2f} s.")
    else:
        print("Obie wersje wykonały się w tym samym czasie.")


if __name__ == "__main__":
    main()

=== Pobieranie sekwencyjne ===
1. In 1888, more than 300,000 mummified cats were found an Egyptian cemetery. They were stripped of their wrappings and carted off to be used by farmers in England and the U.S. for fertilizer.
2. A cat rubs against people not only to be affectionate but also to mark out its territory with scent glands around its face. The tail area and paws also carry the cat’s scent.
3. Among many other diseases, cats can suffer from anorexia, senility, feline AIDS and acne.
4. The richest cat is Blackie who was left £15 million by his owner, Ben Rea.
5. A cat called Dusty has the known record for the most kittens. She had more than 420 kittens in her lifetime.
6. A cat has 230 bones in its body. A human has 206. A cat has no collarbone, so it can fit through any opening the size of its head.
7. Cats have 30 vertebrae (humans have 33 vertebrae during early development; 26 after the sacral and coccygeal regions fuse)
8. The lightest cat on record is a blue point Himalayan

### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [6]:
# Miejsce na rozwiązanie zadania 2
import queue
import threading
import time

NUMBERS_TO_GENERATE = 20
q = queue.Queue()

EVEN_SENTINEL = "STOP_EVEN"
ODD_SENTINEL = "STOP_ODD"


def producer():
    for number in range(NUMBERS_TO_GENERATE):
        q.put(number)
        print(f"Producent dodał: {number}")
        time.sleep(0.05)

    q.put(EVEN_SENTINEL)
    q.put(ODD_SENTINEL)
    print("Producent zakończył działanie.")


def even_consumer():
    while True:
        item = q.get()

        if item == EVEN_SENTINEL:
            print("Konsument parzystych kończy działanie.")
            q.task_done()
            break

        if item == ODD_SENTINEL:
            q.put(item)
            q.task_done()
            time.sleep(0.01)
            continue

        if item % 2 == 0:
            print(f"Konsument parzystych pobrał: {item}")
            q.task_done()
        else:
            q.put(item)
            q.task_done()
            time.sleep(0.01)


def odd_consumer():
    while True:
        item = q.get()

        if item == ODD_SENTINEL:
            print("Konsument nieparzystych kończy działanie.")
            q.task_done()
            break

        if item == EVEN_SENTINEL:
            q.put(item)
            q.task_done()
            time.sleep(0.01)
            continue

        if item % 2 != 0:
            print(f"Konsument nieparzystych pobrał: {item}")
            q.task_done()
        else:
            q.put(item)
            q.task_done()
            time.sleep(0.01)


producer_thread = threading.Thread(target=producer)
even_thread = threading.Thread(target=even_consumer)
odd_thread = threading.Thread(target=odd_consumer)

producer_thread.start()
even_thread.start()
odd_thread.start()

producer_thread.join()
q.join()
even_thread.join()
odd_thread.join()

print("Program zakończył działanie.")

Producent dodał: 0Konsument parzystych pobrał: 0

Producent dodał: 1
Konsument nieparzystych pobrał: 1
Producent dodał: 2
Konsument parzystych pobrał: 2
Producent dodał: 3
Konsument nieparzystych pobrał: 3
Producent dodał: 4
Konsument parzystych pobrał: 4
Producent dodał: 5
Konsument nieparzystych pobrał: 5
Producent dodał: 6
Konsument parzystych pobrał: 6
Producent dodał: 7
Konsument nieparzystych pobrał: 7
Producent dodał: 8
Konsument parzystych pobrał: 8
Producent dodał: 9
Konsument nieparzystych pobrał: 9
Producent dodał: 10
Konsument parzystych pobrał: 10
Producent dodał: 11
Konsument nieparzystych pobrał: 11
Producent dodał: 12
Konsument parzystych pobrał: 12
Producent dodał: 13
Konsument nieparzystych pobrał: 13
Producent dodał: 14
Konsument parzystych pobrał: 14
Producent dodał: 15Konsument nieparzystych pobrał: 15

Producent dodał: 16
Konsument parzystych pobrał: 16
Producent dodał: 17
Konsument nieparzystych pobrał: 17
Producent dodał: 18
Konsument parzystych pobrał: 18
Produ

### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [7]:
# Miejsce na rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    numbers = list(range(1, 10001))

    start_time = time.time()

    with multiprocessing.Pool(processes=4) as pool:
        results = pool.map(calculate_power_sum, numbers)

    end_time = time.time()

    print("Pierwsze 10 wyników:")
    for i, result in enumerate(results[:10], start=1):
        print(f"{i} -> {result}")

    print(f"\nObliczono {len(results)} liczb.")
    print(f"Czas wykonania: {end_time - start_time:.2f} s")
    pass

Pierwsze 10 wyników:
1 -> 100
2 -> 2535301200456458802993406410750
3 -> 773066281098016996554691694648431909053161283000
4 -> 2142584059011987034055949456454883470029603991710390447068500
5 -> 9860761315262647567646607066034827870915080438862787559628486633300780
6 -> 783982348200085087316028320589669384644572452567545845851686359643396569772850
7 -> 3773555927895550989902089063950252946000070398722062967756211219956973369576416070000
8 -> 2328041115810841241449652215325003612630249592761070000727017656405007199729527664209597000
9 -> 298815737485359115506128987290252080182887634235068807971396831956479052263964955868682786424500
10 -> 11111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111110

Obliczono 10000 liczb.
Czas wykonania: 0.41 s
